# 04a -- ECG Forecasting: WaveNet (4.9s -> 4.9s) -- Fixed v3### Every bug from v2 identified and fixed| # | Bug in v2 | Fix in v3 ||---|-----------|-----------|| 1 | Symmetric padding -- spikes smeared, model sees future | **Left-only (causal) padding** -- no temporal leakage || 2 | No skip connections -- early blocks starved of gradient | **WaveNet skip sum** -- every block contributes directly || 3 | SpikeWeightedMSE uses global std -- one loud lead corrupts thresholds | **Per-lead std** -- each lead detected independently || 4 | CosineAnnealingLR + early stop mismatch | **OneCycleLR** -- warm-up then anneal in whatever epochs run || 5 | 9 blocks, dilation 256 -- huge wasted padding | **7 blocks, dilation up to 64** -- RF=763>490, zero waste || 6 | batch=64, lr=3e-4 | **batch=128, lr=5e-4** -- tuned |

In [ ]:
# CELL 1 -- IMPORTSimport os, pickle, warningsimport numpy as npimport matplotlib.pyplot as pltfrom tqdm.auto import tqdmwarnings.filterwarnings('ignore')import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoaderfrom torch.optim.lr_scheduler import OneCycleLRfrom sklearn.metrics import mean_absolute_error, mean_squared_errorSEED = 42torch.manual_seed(SEED); np.random.seed(SEED)if torch.cuda.is_available():    torch.cuda.manual_seed_all(SEED)    torch.backends.cudnn.deterministic = True    torch.backends.cudnn.benchmark     = FalseDEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"PyTorch : {torch.__version__}")print(f"Device  : {DEVICE}")if DEVICE.type == 'cuda':    print(f"GPU     : {torch.cuda.get_device_name(0)}")print("OK Imports ready")

In [ ]:
# CELL 2 -- LOAD DATASAVE_DIR = os.path.join('..', 'data', 'processed')FIG_DIR  = os.path.join('..', 'reports', 'figures', 'cnn_lstm_v3')CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')os.makedirs(FIG_DIR,  exist_ok=True)os.makedirs(CKPT_DIR, exist_ok=True)X_train = np.load(os.path.join(SAVE_DIR, 'X_train.npy'))y_train = np.load(os.path.join(SAVE_DIR, 'y_train.npy'))X_val   = np.load(os.path.join(SAVE_DIR, 'X_val.npy'))y_val   = np.load(os.path.join(SAVE_DIR, 'y_val.npy'))X_test  = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))y_test  = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:    cfg = pickle.load(f)LEAD_NAMES = cfg['lead_names']FS         = cfg['sampling_rate']INPUT_LEN  = cfg['input_len']HORIZON    = cfg['horizon']N_LEADS    = cfg['n_leads']print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")print(f"Input   : {INPUT_LEN} samples = {INPUT_LEN/FS:.2f}s")print(f"Horizon : {HORIZON}  samples = {HORIZON/FS:.2f}s")assert y_train.ndim == 3,           "y must be 3-D (N, HORIZON, 12)"assert y_train.shape[1] == HORIZON, "horizon mismatch"assert y_train.shape[2] == N_LEADS, "leads mismatch"print("OK Data loaded and verified")

In [ ]:
# CELL 3 -- DATASET & DATALOADERSclass ECGForecastDataset(Dataset):    def __init__(self, X, y, channel_first=False):        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32))        self.channel_first = channel_first    def __len__(self): return len(self.X)    def __getitem__(self, idx):        x, y = self.X[idx], self.y[idx]        if self.channel_first: x = x.permute(1, 0)        return x, ydef make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te,                 channel_first=False, batch_train=128, batch_eval=256):    kw = dict(num_workers=0, pin_memory=(DEVICE.type == 'cuda'))    tr = DataLoader(ECGForecastDataset(X_tr, y_tr, channel_first),                    batch_size=batch_train, shuffle=True,  drop_last=True,  **kw)    vl = DataLoader(ECGForecastDataset(X_v,  y_v,  channel_first),                    batch_size=batch_eval,  shuffle=False, **kw)    te = DataLoader(ECGForecastDataset(X_te, y_te, channel_first),                    batch_size=batch_eval,  shuffle=False, **kw)    return tr, vl, tecnn_tr, cnn_vl, cnn_te = make_loaders(    X_train, y_train, X_val, y_val, X_test, y_test,    channel_first=True, batch_train=128, batch_eval=256)xb, yb = next(iter(cnn_tr))print(f"Train batch : x={xb.shape}  y={yb.shape}")print(f"Batches     : train={len(cnn_tr)} | val={len(cnn_vl)} | test={len(cnn_te)}")print("OK DataLoaders ready -- channel-first (B, 12, T)")

In [ ]:
# CELL 4 -- MODEL ARCHITECTURE# FIX 1 -- Left-only (causal) padding# FIX 2 -- WaveNet skip connections# FIX 5 -- 7 blocks, dilation capped at 64class CausalDilatedBlock(nn.Module):    def __init__(self, channels, dilation, kernel_size=7, dropout=0.10):        super().__init__()        self.causal_pad = (kernel_size - 1) * dilation        self.conv_tanh = nn.Conv1d(channels, channels, kernel_size,                                   dilation=dilation, padding=0, bias=False)        self.conv_gate = nn.Conv1d(channels, channels, kernel_size,                                   dilation=dilation, padding=0, bias=False)        self.bn_t  = nn.BatchNorm1d(channels)        self.bn_g  = nn.BatchNorm1d(channels)        self.skip_proj = nn.Conv1d(channels, channels, 1, bias=False)        self.res_proj  = nn.Conv1d(channels, channels, 1, bias=False)        self.drop = nn.Dropout(dropout)    def forward(self, x):        xp    = F.pad(x, (self.causal_pad, 0))        t_out = torch.tanh(   self.bn_t(self.conv_tanh(xp)))        g_out = torch.sigmoid(self.bn_g(self.conv_gate(xp)))        gated = self.drop(t_out * g_out)        skip  = self.skip_proj(gated)        res   = x + self.res_proj(gated)        return res, skipclass WaveNetForecaster(nn.Module):    DILATIONS   = [1, 2, 4, 8, 16, 32, 64]    CHANNELS    = 128    KERNEL_SIZE = 7    def __init__(self, n_leads=12, horizon=490, dropout=0.10):        super().__init__()        self.horizon = horizon        self.n_leads = n_leads        C = self.CHANNELS        self.input_proj = nn.Sequential(            nn.Conv1d(n_leads, C, kernel_size=1, bias=False),            nn.BatchNorm1d(C), nn.GELU())        self.blocks = nn.ModuleList([            CausalDilatedBlock(C, d, self.KERNEL_SIZE, dropout)            for d in self.DILATIONS        ])        self.output_head = nn.Sequential(            nn.GELU(),            nn.Conv1d(C, C // 2, kernel_size=1, bias=False),            nn.BatchNorm1d(C // 2), nn.GELU(),            nn.Conv1d(C // 2, n_leads, kernel_size=1))    def forward(self, x):        out      = self.input_proj(x)        skip_sum = torch.zeros_like(out)        for block in self.blocks:            out, skip = block(out)            skip_sum  = skip_sum + skip        pred = self.output_head(skip_sum)        return pred.permute(0, 2, 1)    def enable_mc_dropout(self):        self.eval()        for m in self.modules():            if isinstance(m, nn.Dropout): m.train()K  = WaveNetForecaster.KERNEL_SIZED  = WaveNetForecaster.DILATIONSrf = (K - 1) * sum(D) + 1print(f"Receptive field : {rf} steps = {rf/100:.2f}s")print(f"RF covers input : {rf >= 490}  ({'OK' if rf >= 490 else 'FAIL'})")print("OK WaveNetForecaster -- Causal padding + Skip connections + No pooling + 7 blocks")

In [ ]:
# CELL 5 -- INSTANTIATE & SHAPE CHECKmodel    = WaveNetForecaster(    n_leads=N_LEADS, horizon=HORIZON, dropout=0.10).to(DEVICE)n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)with torch.no_grad():    dummy = torch.randn(4, N_LEADS, INPUT_LEN).to(DEVICE)    out   = model(dummy)    print(f"Forward pass : {tuple(dummy.shape)} -> {tuple(out.shape)}")    assert out.shape == (4, HORIZON, N_LEADS)print(f"Parameters   : {n_params:,}")print(f"Device       : {DEVICE}")print("OK Model ready")

In [ ]:
# CELL 6 -- SPIKE-WEIGHTED MSE LOSS (per-lead std fix)# FIX 3: per-lead std instead of global std.SPIKE_W     = 6.0SPIKE_SIGMA = 1.2class SpikeWeightedMSE(nn.Module):    def __init__(self, spike_weight=SPIKE_W, spike_sigma=SPIKE_SIGMA):        super().__init__()        self.spike_weight = spike_weight        self.spike_sigma  = spike_sigma    def forward(self, pred, target):        std  = target.std(dim=1, keepdim=True).clamp(min=1e-6)        mask = (target.abs() > self.spike_sigma * std).float()        w    = 1.0 + (self.spike_weight - 1.0) * mask        return (w * (pred - target) ** 2).mean()criterion = SpikeWeightedMSE(spike_weight=SPIKE_W, spike_sigma=SPIKE_SIGMA)print(f'SpikeWeightedMSE  spike_weight={SPIKE_W}x  threshold={SPIKE_SIGMA}sigma  (per lead)')print("OK Loss ready")

In [ ]:
# CELL 7 -- TRAINING ENGINE# FIX 4: OneCycleLR instead of CosineAnnealing + early stop.def train_epoch(model, loader, optimizer, scheduler, device):    model.train()    total_loss = 0.0    for xb, yb in tqdm(loader, desc='Train', leave=False, ncols=88):        xb = xb.to(device, non_blocking=True)        yb = yb.to(device, non_blocking=True)        optimizer.zero_grad(set_to_none=True)        pred = model(xb)        loss = criterion(pred, yb)        loss.backward()        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)        optimizer.step()        scheduler.step()        total_loss += loss.item() * len(xb)    return total_loss / len(loader.dataset)@torch.no_grad()def eval_epoch(model, loader, device):    model.eval()    total_loss, preds, targets = 0.0, [], []    for xb, yb in loader:        xb = xb.to(device, non_blocking=True)        yb = yb.to(device, non_blocking=True)        pred = model(xb)        loss = criterion(pred, yb)        total_loss += loss.item() * len(xb)        preds.append(pred.cpu().numpy())        targets.append(yb.cpu().numpy())    return (total_loss / len(loader.dataset),            np.concatenate(preds,   axis=0),            np.concatenate(targets, axis=0))def train_model(model, tr_loader, vl_loader,                n_epochs=80, max_lr=5e-4, patience=20):    steps_per_epoch = len(tr_loader)    optimizer = torch.optim.AdamW(        model.parameters(), lr=max_lr / 25,        weight_decay=1e-4, eps=1e-8)    scheduler = OneCycleLR(        optimizer, max_lr=max_lr, epochs=n_epochs,        steps_per_epoch=steps_per_epoch, pct_start=0.15,        anneal_strategy='cos', div_factor=25.0, final_div_factor=1e4)    best_val, no_improve = float('inf'), 0    ckpt    = os.path.join(CKPT_DIR, 'WaveNet_v3_best.pt')    history = {'train_loss': [], 'val_loss': [], 'lr': []}    sep = '-' * 72    print(f'\n{sep}')    print(f'  >>> NEW MODEL: WaveNet v3 | Causal + Skip + PerLeadSpike + OneCycleLR <<<')    print(f'  WaveNet v3  |  {INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s  |  {n_params:,} params')    print(f'  Loss: SpikeWeightedMSE({SPIKE_W}x, per-lead)  |  OneCycleLR(max={max_lr:.0e})')    print(f'  batch={tr_loader.batch_size}  steps/ep={steps_per_epoch}  patience={patience}')    print(sep)    pbar = tqdm(range(1, n_epochs + 1), desc='Epochs', unit='ep', ncols=88)    for ep in pbar:        tr_loss       = train_epoch(model, tr_loader, optimizer, scheduler, DEVICE)        vl_loss, _, _ = eval_epoch(model, vl_loader, DEVICE)        cur_lr        = scheduler.get_last_lr()[0]        history['train_loss'].append(tr_loss)        history['val_loss'].append(vl_loss)        history['lr'].append(cur_lr)        is_best = vl_loss < best_val        if is_best:            best_val = vl_loss; no_improve = 0            torch.save(model.state_dict(), ckpt)        else:            no_improve += 1        pbar.set_postfix(tr=f'{tr_loss:.4f}', vl=f'{vl_loss:.4f}',                         lr=f'{cur_lr:.1e}', pat=no_improve)        if ep % 5 == 0 or is_best:            tqdm.write(                f'  ep {ep:3d}  train={tr_loss:.5f}  val={vl_loss:.5f}'                f'  lr={cur_lr:.2e}'                f'{"  * best" if is_best else f"  (no-imp {no_improve}/{patience})"}')        if no_improve >= patience:            tqdm.write(f'  Early stop at ep {ep}   best val={best_val:.6f}')            break    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))    print(f'\n  Loaded best checkpoint  val={best_val:.6f}')    print(f'{sep}\n')    return historyprint("OK Training engine ready")

In [ ]:
# CELL 8 -- TRAINhistory = train_model(    model, cnn_tr, cnn_vl,    n_epochs = 80,    max_lr   = 5e-4,    patience = 20)

In [ ]:
# CELL 9 -- TRAINING HISTORYfig, axes = plt.subplots(1, 2, figsize=(14, 5))ep_range  = range(1, len(history['train_loss']) + 1)axes[0].plot(ep_range, history['train_loss'],             color='#0ea5e9', lw=2, label='Train', marker='o', ms=3)axes[0].plot(ep_range, history['val_loss'],             color='#ef4444', lw=2, label='Val', marker='s', ms=3, ls='--')best_ep = int(np.argmin(history['val_loss'])) + 1axes[0].axvline(best_ep, color='gold', ls=':', lw=2, label=f'Best ep {best_ep}')axes[0].set_title(f'WaveNet v3 -- Loss ({INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s)',                  fontsize=13, fontweight='bold')axes[0].set_xlabel('Epoch'); axes[0].set_ylabel(f'SpikeWeightedMSE ({SPIKE_W}x)')axes[0].legend(); axes[0].grid(True, alpha=0.3)axes[1].plot(ep_range, history['lr'], color='#10b981', lw=2, marker='o', ms=3)axes[1].set_title('OneCycleLR Schedule', fontsize=13, fontweight='bold')axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Learning Rate')axes[1].set_yscale('log'); axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.savefig(os.path.join(FIG_DIR, '01_training_history.png'), dpi=150, bbox_inches='tight')plt.show()print("OK Saved 01_training_history.png")

In [ ]:
# CELL 10 -- TEST EVALUATIONtest_loss, test_preds, test_targets = eval_epoch(model, cnn_te, DEVICE)print(f'Test SpikeWeightedMSE : {test_loss:.6f}')mae_per_lead, rmse_per_lead = [], []for i in range(N_LEADS):    p = test_preds[:,  :, i].flatten()    t = test_targets[:, :, i].flatten()    mae_per_lead.append( mean_absolute_error(t, p))    rmse_per_lead.append(np.sqrt(mean_squared_error(t, p)))mae_macro  = np.mean(mae_per_lead)rmse_macro = np.mean(rmse_per_lead)print(f"\n{'Lead':>6s}   {'MAE':>8s}   {'RMSE':>8s}")print('-' * 30)for i, name in enumerate(LEAD_NAMES):    print(f"  {name:>4s}   {mae_per_lead[i]:8.4f}   {rmse_per_lead[i]:8.4f}")print('-' * 30)print(f"  {'Macro':>4s}   {mae_macro:8.4f}   {rmse_macro:8.4f}")print('\nOK Evaluation complete')

In [ ]:
# CELL 11 -- PER-LEAD RMSEfig, ax = plt.subplots(figsize=(13, 6))colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, N_LEADS))bars   = ax.bar(np.arange(N_LEADS), rmse_per_lead, width=0.62,                color=colors, alpha=0.88, edgecolor='black', lw=0.5)ax.axhline(rmse_macro, color='#facc15', ls='--', lw=2.5,           label=f'Macro RMSE: {rmse_macro:.4f} mV')for bar, val in zip(bars, rmse_per_lead):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')ax.set_xticks(np.arange(N_LEADS))ax.set_xticklabels(LEAD_NAMES, fontsize=11, fontweight='bold')ax.set_ylabel('RMSE (mV)', fontsize=12)ax.set_title(f'WaveNet v3 -- Per-Lead RMSE', fontsize=14, fontweight='bold')ax.legend(fontsize=11); ax.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.savefig(os.path.join(FIG_DIR, '02_per_lead_rmse.png'), dpi=150, bbox_inches='tight')plt.show()print("OK Saved 02_per_lead_rmse.png")

In [ ]:
# CELL 12 -- PREDICTED vs ACTUALN_ROWS       = 4lead_indices = [0, 1, 2, 6]t_axis       = np.arange(HORIZON) / FSfig, axes = plt.subplots(N_ROWS, 4, figsize=(20, 14))for row in range(N_ROWS):    for col, li in enumerate(lead_indices):        ax     = axes[row, col]        actual = test_targets[row, :, li]        pred   = test_preds[row,   :, li]        rmse_i = np.sqrt(mean_squared_error(actual, pred))        ax.plot(t_axis, actual, color='#0ea5e9', lw=1.8, label='Actual', alpha=0.92)        ax.plot(t_axis, pred,   color='#ef4444', lw=1.4, label='Predicted', alpha=0.88, ls='--')        ax.set_title(f'{LEAD_NAMES[li]}  RMSE={rmse_i:.3f}', fontsize=10, fontweight='bold')        ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('mV', fontsize=8)        ax.grid(True, alpha=0.25)        if col == 0 and row == 0: ax.legend(fontsize=8)fig.suptitle(f'WaveNet v3: Predicted vs Actual', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig(os.path.join(FIG_DIR, '03_predictions_overlay.png'), dpi=150, bbox_inches='tight')plt.show()print("OK Saved 03_predictions_overlay.png")

In [ ]:
# CELL 13 -- SPIKE TRACKINGspike_mae_l, base_mae_l = [], []bm_l, sm_l = [], []for i in range(N_LEADS):    actual  = test_targets[:, :, i]    pred    = test_preds[:,   :, i]    thresh  = actual.std() * SPIKE_SIGMA    sp_m = np.abs(actual) > thresh    bm   = ~sp_m    sm_mae = np.abs(actual[sp_m] - pred[sp_m]).mean() if sp_m.sum() else 0.0    bm_mae = np.abs(actual[bm]  - pred[bm]).mean()  if bm.sum()  else 0.0    spike_mae_l.append(sm_mae); base_mae_l.append(bm_mae)    sm_l.append(sm_mae); bm_l.append(bm_mae)spike_mean = np.mean(spike_mae_l)base_mean  = np.mean(base_mae_l)ratio      = spike_mean / (base_mean + 1e-9)print(f'Baseline MAE (flat regions) : {base_mean:.4f} mV')print(f'Spike MAE   (|x| > {SPIKE_SIGMA}sigma)   : {spike_mean:.4f} mV')print(f'Spike / baseline ratio      : {ratio:.2f}x')if   ratio < 2.5: print('Excellent spike tracking')elif ratio < 4.0: print('Good spike tracking')elif ratio < 6.0: print('Partial -- try more epochs')else:             print('FAIL -- Spikes still poor')fig, axes = plt.subplots(1, 2, figsize=(16, 6))x = np.arange(N_LEADS)axes[0].bar(x - 0.2, bm_l, 0.38, label=f'Baseline ({base_mean:.3f})',            color='#0ea5e9', alpha=0.85, edgecolor='black')axes[0].bar(x + 0.2, sm_l, 0.38, label=f'Spike ({spike_mean:.3f})',            color='#ef4444', alpha=0.85, edgecolor='black')axes[0].set_xticks(x); axes[0].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')axes[0].set_ylabel('MAE (mV)'); axes[0].set_title(f'Spike/Base ratio={ratio:.2f}x', fontweight='bold')axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='y')af = test_targets[:,:,1].flatten(); pf = test_preds[:,:,1].flatten()idx = np.random.choice(len(af), min(10000, len(af)), replace=False)axes[1].scatter(af[idx], pf[idx], alpha=0.12, s=3, color='#10b981')mn = min(af.min(), pf.min()); mx = max(af.max(), pf.max())axes[1].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect')axes[1].set_xlabel('Actual (mV)'); axes[1].set_ylabel('Predicted (mV)')axes[1].set_title('Lead II scatter', fontweight='bold')axes[1].legend(); axes[1].grid(True, alpha=0.3)plt.tight_layout()plt.savefig(os.path.join(FIG_DIR, '04_spike_tracking.png'), dpi=150, bbox_inches='tight')plt.show()print("OK Saved 04_spike_tracking.png")

In [ ]:
# CELL 14 -- FLAT-LINE CHECKpred_std = test_preds.std(axis=0)true_std = test_targets.std(axis=0)t_axis   = np.arange(HORIZON) / FSfig, axes = plt.subplots(3, 4, figsize=(18, 10))flat_leads = []for i, (ax, name) in enumerate(zip(axes.flatten(), LEAD_NAMES)):    ax.plot(t_axis, true_std[:, i], color='#0ea5e9', lw=1.4, label='Actual std')    ax.plot(t_axis, pred_std[:, i], color='#ef4444', lw=1.4, label='Pred std', ls='--')    ax.set_title(f'Lead {name}', fontweight='bold', fontsize=10)    ax.set_xlabel('Time (s)', fontsize=8); ax.set_ylabel('Std (mV)', fontsize=8)    ax.grid(alpha=0.25)    if i == 0: ax.legend(fontsize=8)    if pred_std[:, i].min() < 0.01:        ax.set_facecolor('#fff0f0'); flat_leads.append(name)        ax.set_title(f'Lead {name} FLAT?', color='red', fontweight='bold')fig.suptitle('WaveNet v3 -- Flat-line Check', fontsize=13, fontweight='bold')plt.tight_layout()plt.savefig(os.path.join(FIG_DIR, '05_flatline_check.png'), dpi=150, bbox_inches='tight')plt.show()if flat_leads:    print(f'WARNING: Flat leads: {flat_leads}')else:    print(f'OK No flat lines. Min pred std = {pred_std.min():.4f} mV')

In [ ]:
# CELL 15 -- SAVE RESULTSresults = {    'model': 'WaveNet_v3_fixed',    'horizon_s': HORIZON/FS, 'input_s': INPUT_LEN/FS,    'n_parameters': n_params, 'test_sw_mse': float(test_loss),    'mae_per_lead': mae_per_lead, 'mae_macro': float(mae_macro),    'rmse_per_lead': rmse_per_lead, 'rmse_macro': float(rmse_macro),    'spike_ratio': float(ratio), 'spike_mean_mae': float(spike_mean),    'base_mean_mae': float(base_mean),    'history': history, 'lead_names': LEAD_NAMES,    'test_preds': test_preds, 'test_targets': test_targets,}path = os.path.join(CKPT_DIR, 'WaveNet_v3_results.pkl')with open(path, 'wb') as f:    pickle.dump(results, f)print(f'\n{"="*62}')print(f'  WaveNet v3 FINAL   ({INPUT_LEN/FS:.1f}s -> {HORIZON/FS:.1f}s)')print(f'{"="*62}')print(f'  Parameters    : {n_params:,}')print(f'  Test SW-MSE   : {test_loss:.6f}')print(f'  Macro MAE     : {mae_macro:.6f} mV')print(f'  Macro RMSE    : {rmse_macro:.6f} mV')print(f'  Spike ratio   : {ratio:.2f}x  (target < 4.0)')print(f'{"="*62}')print('OK WaveNet v3 complete -- all bugs fixed')